# Kafka Streams

## What's covered

- The pitch — a library, not a framework
- The three abstractions — `KStream`, `KTable`, `GlobalKTable`
- Topology — the DAG of processor nodes that becomes your runtime
- The threading model — stream threads, tasks, partitions, how scaling works
- State stores and changelog topics — local state with a durable backup
- Stateless operations — `map`, `filter`, `flatMap`, `branch`
- Stateful aggregations — `groupBy` + `count`/`reduce`/`aggregate`
- Joins — `KStream`-`KStream`, `KStream`-`KTable`, `KTable`-`KTable`, foreign-key joins
- Windowing — tumbling, hopping, sliding, session
- Time semantics — event time vs processing time, grace periods, suppression
- Exactly-once processing — `processing.guarantee=exactly_once_v2`
- Interactive queries — exposing state stores as a read API
- ksqlDB — the SQL layer on top, mentioned and pointed at
- Common gotchas

*All code examples are in Java. Kafka Streams is a JVM-only library — the Python `confluent-kafka` client cannot consume from or produce to a Streams topology directly.*

## The pitch — a library, not a framework

Stream processing usually means standing up a cluster — Flink, Spark Streaming, Storm. Each has its own scheduler, its own deployment model, its own way of failing in production at 3 AM.

**Kafka Streams is just a JAR.** You add it to a regular JVM application (Spring Boot service, Micronaut app, whatever), build a topology in your `main()`, call `streams.start()`, and you have a stream processor — running inside your service, scaled by running more copies of your service, deployed by your existing deploy pipeline. No new cluster, no new scheduler, no new on-call rotation.

What you get for free:

- **State that survives failures.** Local RocksDB stores, durably backed by Kafka changelog topics. Lose an instance, another picks up the partitions and re-hydrates state from the changelog.
- **Exactly-once.** End-to-end exactly-once processing through the consume-process-produce pattern, built on the transactional producer from notebook 02.
- **Scaling.** Add more application instances; Kafka's consumer-group rebalance moves partitions (and their state) around. Same horizontal-scaling story as a regular consumer, just with state attached.

**When to reach for Streams over a plain consumer:** anything stateful — joining two streams, aggregating, windowing, deduplicating across records. **When *not* to:** stateless transforms (a plain consumer is simpler) or anything cross-language (Streams is JVM only — Python/Go services should write a regular consumer, or use a different engine).

**Streams vs Flink, briefly.** Flink is more powerful, has richer windowing semantics, and runs as its own cluster — better for complex CEP, multi-stream joins with elaborate watermarks, batch-and-stream unified work. Streams is *good enough* for the vast majority of stream-processing-in-a-microservice use cases, and the operational cost is zero compared to Flink. Most teams pick Streams first and only graduate to Flink when they hit a limit.

## The three abstractions

Kafka Streams models a topic in three different ways. Picking the right one for each topic is most of the design work.

**`KStream<K, V>` — an unbounded sequence of independent records.** Each record stands alone. The natural shape for *event streams* — `payments.created`, `clicks`, `orders.shipped`. Two records with the same key are two distinct events, not an update.

**`KTable<K, V>` — a continuously updated table.** Records with the same key replace each other. Tombstones (`null` values) delete keys. The natural shape for *state* — `users.profile`, `inventory.level`. Reads of a `KTable` give you the current value per key, materialized in a local state store. Built directly on top of compacted topics (notebook 04).

**`GlobalKTable<K, V>` — a `KTable` replicated to every instance in full.** Every application instance holds a complete copy. Expensive for memory but eliminates the co-partitioning requirement on joins (next section). Used for small dimension tables — country codes, feature flags, currency conversion rates.

| | `KStream` | `KTable` | `GlobalKTable` |
|---|---|---|---|
| Records with same key | Independent events | Updates (newer replaces older) | Updates, broadcast |
| Local materialization | None (pass-through) | Per-task state store | Full copy on every instance |
| Backing topic policy | `delete` | `compact` | `compact` |
| Used for | Events | State / dimension tables | Small, broadcast dimensions |

**The mental shortcut:** "is this *what happened* or *what is*?" `KStream` for what happened, `KTable` for what is.

## Topology — the runtime as a DAG

Your code constructs a **topology** — a directed graph of **processor nodes** connected by edges. Source nodes read from Kafka topics, sink nodes write back, and processor nodes in between do your filters, joins, aggregations.

```text
  source: payments     ──► filter(amount>0) ──► groupByKey ──► count ──► sink: payment-counts
                                                  │
                                                  └── materialized as state store "counts"
```

Two things to keep in mind:

- **`stream.print()` and friends are convenience sinks**, not debug output — they appear as nodes in the topology and run on every record.
- **Inspect the topology before you ship it.** `streams.topology().describe()` prints a text rendering of every node and the sub-topologies they belong to. A surprising topology is almost always a bug — joins that should be co-partitioned but aren't, repartitions you didn't expect, missing materializations.

## The threading model

A Streams application contains:

- **Stream threads** — configured via `num.stream.threads`. Each is a JVM thread that runs one or more tasks. Default is 1.
- **Tasks** — one per *(sub-topology, source partition)* pair. A task owns a fixed set of partitions across all source topics it touches, plus its slice of every state store.
- **Sub-topologies** — Streams splits your topology into independent units wherever a repartition happens. Tasks in different sub-topologies are independent of each other.

**The scaling rule:** **maximum parallelism = number of partitions of your source topics.** A topology reading from a 12-partition topic can run at most 12 tasks. Spread those across instances and threads however you like — 1 instance × 12 threads, 12 instances × 1 thread, 6 × 2, all equivalent. Adding a 13th thread or instance gets you one idle worker.

Add more instances of your application → rebalance moves tasks (and their state) to balance load. Same model as a consumer group, with the extra wrinkle that state stores migrate too.

## State stores and changelog topics

Stateful operations (`count`, `reduce`, `aggregate`, joins) need somewhere to keep state per key. Streams uses a **local state store** — by default a RocksDB instance on the task's host — and durably backs every change to a **changelog topic** in Kafka.

```text
  process(record) ──► update local store ──► append change to changelog topic
                            │
                            └── reads serve from local store (fast)
```

Two things you should know:

- **Changelog topics are auto-created**, named `<app-id>-<store-name>-changelog`, and **must be compacted** so they hold only the latest value per key. Streams configures this automatically when it creates them.
- **On recovery, Streams replays the changelog into a fresh local store.** A new instance picking up a task reads its slice of the changelog from offset 0 to the current end, rebuilding the RocksDB state, before processing any new records. For a large state store, recovery can take minutes — which is why **standby replicas** (`num.standby.replicas`) exist: hot copies of the state on other instances, kept warm by tailing the changelog continuously.

**Standby replicas are the standard production tuning knob for state-heavy apps.** `num.standby.replicas=1` means every task has a warm copy elsewhere; failover is seconds, not minutes.

## Setup — the dependencies

Maven coordinates for a Streams application:

```xml
<dependency>
  <groupId>org.apache.kafka</groupId>
  <artifactId>kafka-streams</artifactId>
  <version>3.8.0</version>
</dependency>

<!-- Optional but typical: Avro SerDes for Schema Registry -->
<dependency>
  <groupId>io.confluent</groupId>
  <artifactId>kafka-streams-avro-serde</artifactId>
  <version>7.7.1</version>
</dependency>
```

Minimum configuration. The `application.id` is the analogue of `group.id` for a consumer — it's the identity Streams uses for consumer-group membership, state-store changelog naming, internal topic naming, and exactly-once transaction coordination. **Pick it carefully and never change it casually.**

```java
Properties props = new Properties();
props.put(StreamsConfig.APPLICATION_ID_CONFIG,      "payments-aggregator-v1");
props.put(StreamsConfig.BOOTSTRAP_SERVERS_CONFIG,   "localhost:9092");
props.put(StreamsConfig.DEFAULT_KEY_SERDE_CLASS_CONFIG,   Serdes.String().getClass());
props.put(StreamsConfig.DEFAULT_VALUE_SERDE_CLASS_CONFIG, Serdes.String().getClass());
props.put(StreamsConfig.PROCESSING_GUARANTEE_CONFIG, StreamsConfig.EXACTLY_ONCE_V2);
props.put(StreamsConfig.NUM_STANDBY_REPLICAS_CONFIG, 1);
```

## Hello Streams — count payments per customer

Read from `payments.created`, group by customer ID, count per customer, write the running count to `payment-counts`. The whole thing is ten lines of topology code.

```java
StreamsBuilder builder = new StreamsBuilder();

KStream<String, String> payments = builder.stream("payments.created");

KTable<String, Long> counts = payments
    .groupByKey()                    // key is already customer_id from the producer
    .count(Materialized.as("payment-counts-store"));

counts.toStream().to("payment-counts", Produced.with(Serdes.String(), Serdes.Long()));

KafkaStreams streams = new KafkaStreams(builder.build(), props);
streams.start();
Runtime.getRuntime().addShutdownHook(new Thread(streams::close));
```

What's happening here:

- **`builder.stream(...)`** declares a source node — a `KStream` reading from `payments.created`.
- **`.groupByKey()`** doesn't shuffle — it asserts "records are already grouped by this key." If we'd called `.groupBy((k, v) -> someOtherKey)`, Streams would have inserted a **repartition topic** (a Kafka topic) to re-shuffle records by the new key before the aggregation. Repartitions are expensive; design your producer keys to avoid them where you can.
- **`.count(Materialized.as("..."))`** materializes the running count into a state store. The `Materialized` is what makes the store queryable by name (interactive queries, below).
- **`counts.toStream().to("payment-counts", ...)`** converts the changelog of updates back into a stream and sinks it to a topic — so downstream services can subscribe to live counts.

Run two copies of this application against the same `application.id` and they automatically share the work: the consumer-group rebalance assigns half the input partitions to each, and each holds half the state.

## Stateless operations

The cheap operations — they don't materialize anything, they don't repartition, they run inline per record:

| Operation | What it does |
|---|---|
| `filter((k, v) -> ...)` | Keep records that pass the predicate |
| `filterNot(...)` | Inverse |
| `map((k, v) -> KeyValue.pair(...))` | Transform key and/or value. **Changes the key — forces repartition before next stateful op** |
| `mapValues(v -> ...)` | Transform value only — no repartition |
| `flatMap(...)` | Zero, one, or many output records per input |
| `flatMapValues(...)` | Same but key-preserving |
| `selectKey((k, v) -> ...)` | Compute a new key. Forces repartition |
| `branch(...)` / `split()` | Split a stream into multiple based on predicates |
| `peek((k, v) -> ...)` | Side-effect (logging, metrics) without changing the stream |

```java
KStream<String, Payment> highValue = payments
    .mapValues(json -> Payment.from(json))          // String -> Payment, key preserved
    .filter((customerId, p) -> p.amount() > 1000)   // keep only high-value
    .peek((k, p) -> meter.mark());                  // metrics side effect
```

**The watch-out: anything that changes the key forces a repartition.** A repartition writes every record to a hidden internal topic and re-reads from it on the other side, doubling the network and storage cost for that segment of the topology. Prefer `mapValues` over `map`, and prefer keying records correctly at the producer.

## Stateful aggregations

`groupByKey()` or `groupBy(...)` returns a `KGroupedStream`; then you pick an aggregation:

- **`count()`** — running count per key.
- **`reduce((v1, v2) -> ...)`** — combine same-typed values: running sum, running max, etc.
- **`aggregate(initializer, aggregator)`** — when output type differs from input. Initialize an accumulator, then fold each value in.

```java
// Sum of payment amounts per customer
KTable<String, Double> totalPerCustomer = payments
    .groupByKey()
    .aggregate(
        () -> 0.0,                                  // initializer
        (customerId, payment, total) -> total + payment.amount(),  // aggregator
        Materialized.<String, Double, KeyValueStore<Bytes, byte[]>>as("customer-totals")
            .withValueSerde(Serdes.Double())
    );
```

Each call updates the local state store *and* appends a record to the changelog topic. Recovery (or a new instance taking over) replays the changelog into a fresh store.

## Joins — four shapes

Streams supports four kinds of join, and the shape decides what semantics you get.

**`KStream`–`KStream` (windowed join).** Two event streams. The join is windowed — "these two events happened within N seconds of each other." Used for things like "impression joined with click within 5 minutes."

```java
KStream<String, Impression> impressions = ...;
KStream<String, Click>      clicks      = ...;

KStream<String, Conversion> conversions = impressions.join(
    clicks,
    (imp, click) -> new Conversion(imp, click),
    JoinWindows.ofTimeDifferenceWithNoGrace(Duration.ofMinutes(5))
);
```

**`KStream`–`KTable` (lookup join).** Event stream joined against a table for enrichment. Non-windowed, asymmetric — every event is enriched with the current table value for its key.

```java
KTable<String, Customer> customers = builder.table("customers");
KStream<String, EnrichedPayment> enriched =
    payments.join(customers, (payment, customer) -> enrich(payment, customer));
```

**`KTable`–`KTable` (changelog join).** Two tables join into a third whose value updates whenever either side changes. Used to combine state from two sources.

**`KStream`/`KTable`–`GlobalKTable` (broadcast lookup).** Like the `KTable` lookup, but the right side is a `GlobalKTable` — fully replicated, no co-partitioning required, the join key doesn't have to match the stream key.

**The co-partitioning rule.** For non-Global joins, both sides must have the *same number of partitions* and the *same partitioner*. If they don't, the join silently produces wrong results — records that should match end up on different tasks. The standard fix is to produce both sides through the same partitioner with the same partition count, or insert a `selectKey` + repartition explicitly.

**Foreign-key joins (`KTable`–`KTable` only).** Newer feature; lets you join two tables on a foreign-key extracted from one side's value, without co-partitioning. Implemented via internal repartition topics behind the scenes. Useful for the classic `orders.customer_id -> customers.id` shape.

## Windowing

Aggregations on infinite streams need a bound. **Windowing** groups records into finite buckets in time:

- **Tumbling** — fixed-size, non-overlapping. "How many payments per minute, exactly."
- **Hopping** — fixed-size, overlapping. "Five-minute counts, recomputed every minute." Defined by `size` + `advance`.
- **Sliding** — windows defined by a max time difference between records — used in `KStream`-`KStream` joins.
- **Session** — variable-length windows defined by a gap of inactivity. "All clicks within 30 minutes of each other form a session."

```java
// Tumbling — count payments per customer per minute
KTable<Windowed<String>, Long> perMinute = payments
    .groupByKey()
    .windowedBy(TimeWindows.ofSizeWithNoGrace(Duration.ofMinutes(1)))
    .count();
```

The key on a windowed `KTable` is wrapped in a `Windowed<K>` — same logical key, plus the window's start/end timestamps. When you sink it to a topic, the output key encodes both.

## Time semantics — event time vs processing time

**The clock that matters for stream processing is the *event time* — when the event actually happened — not the *processing time* — when Streams handles it.** A record produced an hour ago and just consumed should be windowed by its hour-ago timestamp, not by "now."

Streams picks the timestamp via a configurable `TimestampExtractor`:

- **`FailOnInvalidTimestamp`** (default) — use the Kafka record timestamp (the producer's `CreateTime`). Fail on missing/invalid.
- **`UsePartitionTimeOnInvalidTimestamp`** — same, with a graceful fallback.
- **`WallclockTimestampExtractor`** — `System.currentTimeMillis()` — processing time, useful only when you genuinely don't have event time.
- **Custom extractor** — pull the timestamp from inside the record value.

Once you have event time, **out-of-order records** become the question: a record with timestamp `12:01:00` arriving at processing time `12:05:00` is *late*. Two knobs handle that:

- **Window grace period** (`TimeWindows.ofSizeAndGrace(...)`) — how long after a window closes Streams will still accept records into it. Records arriving after the grace period are dropped (and counted in a metric). Default is *zero* in modern Kafka — explicit grace is opt-in.
- **`suppress(...)`** — emit only the final result of a window after grace expires. Without `suppress`, every record inside the window emits an updated aggregate downstream; with it, only the closed window's final value emits.

```java
perMinute
    .suppress(Suppressed.untilWindowCloses(BufferConfig.unbounded()))
    .toStream()
    .to("payment-counts-per-minute");
```

## Exactly-once processing

Set one config:

```java
props.put(StreamsConfig.PROCESSING_GUARANTEE_CONFIG, StreamsConfig.EXACTLY_ONCE_V2);
```

What that turns on:

1. **Transactional producer** with a `transactional.id` derived from `application.id` + task ID.
2. **Consumer with `isolation.level=read_committed`** so the topology sees only committed records from upstream.
3. **Atomic offset-commit + output-produce + state-store changelog write** in a single transaction. Either all three commit or none — no partial state on crash.

**`exactly_once_v2`** (since Kafka 2.6) is much cheaper than the original `exactly_once`: one transaction per stream thread per commit interval instead of one per task. Throughput is close to at-least-once. Use `v2` unless you're stuck on an old broker (`exactly_once_v2` requires brokers on 2.5+).

The cost is a small latency tax (each record waits for its transaction to commit before going downstream — `commit.interval.ms` defaults to 100ms under EOS). For most analytics-shaped pipelines, it's a great trade.

## Interactive queries — read your state from outside

Streams state stores are normally hidden inside the topology. **Interactive queries** expose them: from the same application instance, you can query the current value of any materialized state store by name.

```java
// In an HTTP endpoint somewhere in your application:
ReadOnlyKeyValueStore<String, Long> store = streams.store(
    StoreQueryParameters.fromNameAndType("payment-counts-store", QueryableStoreTypes.keyValueStore())
);
Long count = store.get("CUST0001");   // current running count for this customer
```

**The wrinkle: state is distributed across instances.** Customer `CUST0001` lives on whichever task owns the partition that hashes that key. If your HTTP request hits a different instance, the store on *that* instance doesn't have the key.

**The standard pattern:**

1. The receiving instance asks the metadata layer (`streams.queryMetadataForKey(...)`) which instance owns the key.
2. If it's the local instance, query directly.
3. Otherwise, forward the HTTP request to the owning instance (whose host:port the metadata layer hands back).

Interactive queries turn a Streams app into a low-latency read API on top of your event streams — without needing a separate database. Excellent fit for serving dashboards, customer-facing APIs, and any "give me the latest state" workload.

## ksqlDB — the SQL layer

Brief mention so you can recognize it. **ksqlDB** is a separate server (and product) from Confluent that compiles SQL into Kafka Streams topologies behind the scenes:

```sql
CREATE STREAM payments (customer_id VARCHAR, amount DOUBLE)
  WITH (KAFKA_TOPIC='payments.created', VALUE_FORMAT='AVRO');

CREATE TABLE customer_totals AS
  SELECT customer_id, SUM(amount) AS total
  FROM payments GROUP BY customer_id
  EMIT CHANGES;
```

Same execution model as Streams, just SQL on top. **Out of scope for this curriculum.** When you'd reach for it: when the user writing the pipeline is more comfortable in SQL than Java, or when the pipeline is straightforward enough that Java would be overkill. When you'd reach for Streams directly: anything custom (custom serdes, complex aggregations, integration with other JVM libraries).

## Common gotchas

- **Changing `application.id`.** It's the identity for everything — consumer group, changelog topics, internal repartition topics, EOS transactions. Changing it makes Streams treat your app as brand new: state is rebuilt from scratch, consumer group offsets are gone, you start over. Treat it as immutable.
- **Forgetting to set the value serde when materializing.** Aggregations default to the default value serde, which is often the wrong type (e.g. you're counting into a `Long` but the default is `String`). Explicit `Materialized.with(...)` saves debugging.
- **Unintended repartitions.** `selectKey`, `map` (with new key), and `groupBy` (with new key) all force a repartition. Inspect with `topology.describe()` — if you see a `REPARTITION` node you didn't intend, fix the key at the producer or shift the operation later.
- **Joining streams that aren't co-partitioned.** Silent wrong answers, not loud errors. Verify partition count and partitioner match.
- **Windowed aggregations without `suppress`** when downstream expects the final result. Without it, every record inside the window produces an interim emit downstream.
- **State stores larger than disk.** RocksDB stores can grow large; out of disk takes the whole instance down. Monitor disk usage; reduce store size via TTLs (windowed stores) or move to a smaller key/value.
- **`commit.interval.ms` ignored under EOS v2.** EOS v2 commits at every `commit.interval.ms`, default 100ms. Cranking this up gives you bigger transactions and higher throughput at the cost of higher end-to-end latency.
- **Running Streams against a topic Kafka Connect also writes to with inconsistent serdes.** Same lesson as notebook 06 — one converter, one format, end to end.
- **Standby replicas not configured.** Default is 0. With 0 standbys, an instance failure means the surviving task rebuilds state from offset 0 of the changelog before resuming processing. For any state-heavy app, set `num.standby.replicas >= 1`.

## What's next

Producers, consumers, topic design, schemas, Connect, and Streams — that's the data plane. You can build any event-driven system from here.

- **Notebook 08 — Operations, Security & Performance Tuning.** Closes the curriculum: JMX metrics that matter, SSL/SASL/ACLs, the broker tuning knobs you'll actually touch, partition reassignment, monitoring dashboards, and the operational checklist for a production cluster.

After 08, the curriculum is complete and the loop from "empty topic" to "production stream-processing system" is closed.